In [ ]:
%load_ext autoreload
%autoreload 2
import os
print(os.getcwd())

/home/as/code/ai/susteelaible/nlp


# bert-1
- run pipeline

In [5]:
from nlp import ClimateBERTAnalyzer, analyze_reports

stats = analyze_reports('../data/reports/Baosteel')
# stats = analyze_reports("../data/reports")


CLIMATEBERT ANALYZER
  PDFs: 24
  Cache: cache
  GPU: NVIDIA T1200 Laptop GPU (3.9GB)
  Translation: enabled



...2013_annual_report.pdf [1/10 Extract]:   0%|         | 0/24 [00:00<?, ?pdf/s]

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


...2013_annual_report.pdf [9/10 Commit]:   0%|          | 0/24 [01:32<?, ?pdf/s]

KeyboardInterrupt: 

# bert-2
- run vizualisations

In [ ]:
from nlp import ClimateBERTVisualizer, visualize_results

visualize_results("../cache", "../out")

✅ Loaded: 15 reports, 1 companies, 2013-2020

EXPORTING CSV FILES
   ✓ company_year.csv (8 rows)
   ✓ company_totals.csv (1 companies)
   ✓ yearly_industry.csv (8 years)
   ✓ funnel_company_year.csv (8 rows)

   📁 All CSVs saved to: ../out/

GENERATING PLOTS
   ✓ slide_main.png
   ✓ slide_sentiment_trend.png
   ✓ talk_score_trend.png
   ✓ funnel_trend.png
   ✓ talk_score_per_company.png
   ✓ per_company_components.png
   ✓ per_company_sentiment.png
   ✓ sentiment_all_companies.png
   ✓ n0_funnel.png
   ✓ n0_quality_comparison.png
   ✓ n0_per_company.png
   ✓ n0_gap_analysis.png

   📁 All plots saved to: ../out/

   Generating word frequency plots...

   📊 ALL CHUNKS (top 30):
   environment(1232), iron(766), management(736), development(720), energy(628), technology(609), products(607), protection(559), production(525), industry(522), green(473), emission(453), system(428), reduction(421), base(410)

   🌱 OPPORTUNITY chunks (top 15):
   environment(809), development(574), technology(52

# rag 1

In [ ]:
from nlp import quick_start

# Full pipeline
# pipeline = RAGPipeline()
# pipeline.load_and_process_pdfs()
# pipeline.inspect_pipeline_status()
# pipeline.extract_all_companies()

# Quick start
# Make sure Ollama is running: ollama run llama3.1:8b
pipeline = quick_start('../cache/', ollama_model='llama3.1:8b')


# Load existing
# pipeline = load_existing_pipeline('vector_db/reports')

✓ RAG Pipeline initialized (GPU: NVIDIA T1200 Laptop GPU (3.9GB))
✓ Loaded 15593 chunks from 15 companies (../cache/)


In [5]:
# pipeline.inspect_pipeline_status()
pipeline.extract_all_companies()


EXTRACTING ALL COMPANIES
Total groups: 132
LLM context: 8192 tokens

Extracting: ArcelorMittal (001)
Years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
Groups: 13 | Avg chunks/group: 118.1


  001:   0%|          | 0/13 [00:00<?, ?it/s]

Loading Ollama model: llama3.1:8b


  ✓ Extracted 176 barriers (44% matched), 208 motivators (34% matched)

Extracting: Acerinox (002)
Years: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Groups: 10 | Avg chunks/group: 118.1


  ✓ Extracted 159 barriers (47% matched), 129 motivators (31% matched)

Extracting: Outokumpu (003)
Years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Groups: 12 | Avg chunks/group: 118.1


...

  ✓ Extracted 183 barriers (24% matched), 255 motivators (68% matched)

Extracting: SSAB (005)
Years: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Groups: 12 | Avg chunks/group: 118.1


  ✓ Extracted 144 barriers (49% matched), 158 motivators (26% matched)

Extracting: TataSteelNederland (006)
Years: ['2021', '2022', '2023', '2024']
Groups: 4 | Avg chunks/group: 118.1


  ✓ Extracted 48 barriers (31% matched), 65 motivators (34% matched)

Extracting: Celsa (007)
Years: ['2020', '2021', '2022', '2023', '2024']
Groups: 5 | Avg chunks/group: 118.1


  ✓ Extracted 81 barriers (64% matched), 74 motivators (68% matched)

Extracting: Voestalpine (008)
Years: ['2013', '2015', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
Groups: 9 | Avg chunks/group: 118.1


KeyboardInterrupt: 

# rag 2

In [ ]:
from nlp import TopicModeler, TopicModelConfig, run_topic_modeling_pipeline

# Set True to ignore cached model/embeddings and retrain from scratch
FORCE_RETRAIN = True

config = TopicModelConfig(
    # Embedding model
    embedding_model="sentence-transformers/all-mpnet-base-v2",
    # embedding_model="thenlper/gte-small",
    batch_size=64,

    # UMAP (dimensionality reduction)
    umap_n_neighbors=15,
    umap_n_components=15,
    umap_min_dist=0.2,
    umap_metric='cosine',
    umap_random_state=42,

    # HDBSCAN (clustering)
    hdbscan_min_cluster_size=25,  # Higher = fewer topics
    hdbscan_min_samples=3,        # Lower = less outliers
    hdbscan_metric='euclidean',
    hdbscan_cluster_selection_method='leaf',  # 'eom' or 'leaf'

    # Vectorizer (c-TF-IDF)
    vectorizer_ngram_range=(1, 2),
    vectorizer_min_df=2,
    vectorizer_max_df=0.95,

    # Topic representation
    mmr_diversity=0.3,
    top_n_words=10,
    nr_topics=7,  # Set None for auto, or int to reduce post-hoc
    calculate_probabilities=False,

    # Visualization UMAP (separate 2D projection)
    viz_umap_n_neighbors=10,
    viz_umap_n_components=2,
    viz_umap_min_dist=0.0,

    # LLM for topic labeling
    ollama_model="llama3.1:8b",
    ollama_base_url="http://localhost:11434",
    llm_temperature=0.0,

    # Misc
    verbose=True,
    embeddings_cache_path=None,
)

results = run_topic_modeling_pipeline(
    data_folder="../out",
    output_folder="../out/topics",
    config=config,
    force_retrain=FORCE_RETRAIN
)

# Access results
barriers_df = results['barriers']['df']
motivators_df = results['motivators']['df']

/home/as/code/ai/susteelaible/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
motivators_df